# Step 2: Pair Generator
Simulates the P2P carpooling marketplace. Splits trips into drivers and riders, generates candidate pairs using BallTree spatial indexing, and assigns stochastic acceptance labels.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os, sys
if os.path.basename(os.getcwd()) in ['simulation', 'data', 'models']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import pandas as pd
import numpy as np
import yaml

from src.simulation.feature_eng import calculate_detour_cost, compute_spatial_overlap, calculate_time_overlap

class PairGenerator:
    """
    Simulates a P2P carpooling marketplace from single-passenger taxi trips.
    Splits trips into drivers/riders, pairs them spatially and temporally,
    and generates labeled training data with stochastic acceptance labels.
    """
    def __init__(self, processed_data_path, config_path="config/dev_config.yaml"):
        with open(config_path, "r") as file:
            self.config = yaml.safe_load(file)
        print(f"Loading data from {processed_data_path}...")
        self.df = pd.read_parquet(processed_data_path)
        self.df['tpep_pickup_datetime'] = pd.to_datetime(self.df['tpep_pickup_datetime'])
        self.df['tpep_dropoff_datetime'] = pd.to_datetime(self.df['tpep_dropoff_datetime'])
        self.sim_config = self.config["simulation"]
        
    def filter_commuting_window(self):
        """Filter trips to commute windows. Supports AM + PM via time_windows list in config."""
        windows = self.sim_config.get("time_windows")
        if windows:
            mask = pd.Series(False, index=self.df.index)
            labels = []
            for w in windows:
                mask |= (
                    (self.df['tpep_pickup_datetime'].dt.hour >= w['start_hour']) &
                    (self.df['tpep_pickup_datetime'].dt.hour < w['end_hour'])
                )
                labels.append(f"{w['start_hour']}:00-{w['end_hour']}:00")
            self.df = self.df[mask]
            print(f"Filtering for commute windows: {' + '.join(labels)}")
        else:
            start_hr = self.sim_config["time_window"]["start_hour"]
            end_hr   = self.sim_config["time_window"]["end_hour"]
            print(f"Filtering for commute window: {start_hr}:00 - {end_hr}:00")
            self.df = self.df[
                (self.df['tpep_pickup_datetime'].dt.hour >= start_hr) &
                (self.df['tpep_pickup_datetime'].dt.hour < end_hr)
            ]
        sample_ratio = self.sim_config.get("sample_ratio", 1.0)
        if sample_ratio < 1.0:
            print(f"Downsampling to {sample_ratio*100}% for development.")
            self.df = self.df.sample(frac=sample_ratio, random_state=42)
        print(f"Total trips in window: {len(self.df)}")
        return self.df

    def split_drivers_and_riders(self, driver_ratio=None):
        """Randomly assigns trips as drivers or riders."""
        if driver_ratio is None:
            driver_ratio = self.sim_config.get("driver_ratio", 0.5)
        np.random.seed(42)
        msk = np.random.rand(len(self.df)) < driver_ratio
        self.drivers_df = self.df[msk].copy()
        self.riders_df = self.df[~msk].copy()
        print(f"Assigned {len(self.drivers_df)} Drivers and {len(self.riders_df)} Riders.")

    def generate_candidate_pairs(self, limit=1000):
        """Generates training data. Bipartite matching happens at inference, not during training."""
        import networkx as nx
        print("Starting candidate pair generation...")
        max_detour_km = self.sim_config["max_detour_km"]
        self.drivers_df = self.drivers_df.sort_values('tpep_pickup_datetime')
        self.riders_df = self.riders_df.sort_values('tpep_pickup_datetime')
        final_pairs = []
        d_count = 0
        batch_window_mins = self.sim_config.get("batch_window_mins", 5)
        window_size = pd.Timedelta(minutes=batch_window_mins)
        current_time = self.drivers_df['tpep_pickup_datetime'].min()
        end_time = self.drivers_df['tpep_pickup_datetime'].max()
        
        while current_time < end_time and d_count < limit:
            window_end = current_time + window_size
            batch_drivers = self.drivers_df[(self.drivers_df['tpep_pickup_datetime'] >= current_time) & 
                                            (self.drivers_df['tpep_pickup_datetime'] < window_end)]
            batch_riders = self.riders_df[(self.riders_df['tpep_pickup_datetime'] >= current_time) & 
                                          (self.riders_df['tpep_pickup_datetime'] < window_end)]
            if batch_drivers.empty or batch_riders.empty:
                current_time = window_end
                continue
            from sklearn.neighbors import BallTree
            rider_coords_rad = np.radians(batch_riders[['pickup_latitude', 'pickup_longitude']].values)
            tree = BallTree(rider_coords_rad, metric='haversine')
            EARTH_RADIUS_KM = 6371.0
            SEARCH_RADIUS_KM = self.sim_config.get("search_radius_km", 5.0)
            for driver in batch_drivers.itertuples():
                if d_count >= limit: break
                d_idx = driver.Index
                d_count += 1
                d_start = (driver.pickup_latitude, driver.pickup_longitude)
                d_end = (driver.dropoff_latitude, driver.dropoff_longitude)
                neighbor_indices = tree.query_radius(np.radians([[d_start[0], d_start[1]]]),
                                                     r=SEARCH_RADIUS_KM / EARTH_RADIUS_KM)[0]
                if len(neighbor_indices) == 0:
                    continue
                # shareability_count: how many riders are nearby (Santi et al., PNAS 2014)
                shareability_count = len(neighbor_indices)
                for rider in batch_riders.iloc[neighbor_indices].itertuples():
                    r_idx = rider.Index
                    r_start = (rider.pickup_latitude, rider.pickup_longitude)
                    r_end = (rider.dropoff_latitude, rider.dropoff_longitude)
                    detour, new_dist, d_orig_dist = calculate_detour_cost(d_start, d_end, r_start, r_end)
                    if detour <= max_detour_km:
                        overlap_ratio = compute_spatial_overlap(detour, d_orig_dist)
                        diff_mins = calculate_time_overlap(driver.tpep_pickup_datetime, rider.tpep_pickup_datetime)
                        fare_score = min(rider.fare_amount / max(driver.fare_amount, 1.0), 1.5) / 1.5
                        time_score = max(0, 1.0 - (diff_mins / 15.0))
                        trip_ratio = d_orig_dist / max(d_orig_dist + detour, 0.1)
                        detour_score = 1.0 - (detour / max_detour_km)
                        acceptance_prob = (0.30*detour_score + 0.25*fare_score + 0.25*time_score + 0.20*trip_ratio)
                        is_accepted = 1 if (acceptance_prob + np.random.gumbel(0, 0.1)) > 0.45 else 0
                        fare_ratio = rider.fare_amount / max(driver.fare_amount, 1.0)
                        final_pairs.append({
                            'driver_id': d_idx, 'rider_id': r_idx,
                            'driver_orig_dist': d_orig_dist, 'detour_distance_km': detour,
                            'overlap_ratio': overlap_ratio, 'time_diff_mins': diff_mins,
                            'driver_approx_fare': driver.fare_amount, 'rider_approx_fare': rider.fare_amount,
                            'fare_ratio': fare_ratio,
                            'pickup_hour': driver.tpep_pickup_datetime.hour,
                            'pickup_dow': driver.tpep_pickup_datetime.weekday(),
                            'shareability_count': shareability_count,
                            'label': is_accepted
                        })
            current_time = window_end
            if d_count % 100 == 0:
                print(f"Processed {d_count} drivers. Feasible pairs: {len(final_pairs)}")

        print("Generating negative labels from genuinely infeasible pairs...")
        rng = np.random.RandomState(42)
        negative_pairs = []
        target_negatives = len(final_pairs)
        attempts = 0
        while len(negative_pairs) < target_negatives and attempts < target_negatives * 10:
            attempts += 1
            d_idx = self.drivers_df.sample(random_state=rng).index[0]
            r_idx = self.riders_df.sample(random_state=rng).index[0]
            d_start = (self.drivers_df.loc[d_idx, 'pickup_latitude'], self.drivers_df.loc[d_idx, 'pickup_longitude'])
            d_end = (self.drivers_df.loc[d_idx, 'dropoff_latitude'], self.drivers_df.loc[d_idx, 'dropoff_longitude'])
            r_start = (self.riders_df.loc[r_idx, 'pickup_latitude'], self.riders_df.loc[r_idx, 'pickup_longitude'])
            r_end = (self.riders_df.loc[r_idx, 'dropoff_latitude'], self.riders_df.loc[r_idx, 'dropoff_longitude'])
            detour, _, d_orig_dist = calculate_detour_cost(d_start, d_end, r_start, r_end)
            if detour <= max_detour_km:
                continue
            overlap_ratio = compute_spatial_overlap(detour, d_orig_dist)
            driver_time = self.drivers_df.loc[d_idx, 'tpep_pickup_datetime']
            rider_time = self.riders_df.loc[r_idx, 'tpep_pickup_datetime']
            time_diff = calculate_time_overlap(driver_time, rider_time)
            d_fare = self.drivers_df.loc[d_idx, 'fare_amount']
            r_fare = self.riders_df.loc[r_idx, 'fare_amount']
            negative_pairs.append({
                'driver_id': d_idx, 'rider_id': r_idx,
                'driver_orig_dist': d_orig_dist, 'detour_distance_km': detour,
                'overlap_ratio': overlap_ratio, 'time_diff_mins': time_diff,
                'driver_approx_fare': d_fare, 'rider_approx_fare': r_fare,
                'fare_ratio': r_fare / max(d_fare, 1.0),
                'pickup_hour': driver_time.hour, 'pickup_dow': driver_time.weekday(),
                'shareability_count': 0,  # random pair, not from BallTree batch
                'label': 0
            })
        print(f"Generated {len(negative_pairs)} negative pairs from {attempts} attempts.")
        results_df = pd.DataFrame(final_pairs + negative_pairs)
        print(f"Generated {len(results_df)} total labeled pairs (Balanced 1s and 0s).")
        results_df.to_parquet("data/simulated/candidate_pairs.parquet", index=False)
        print("Saved to data/simulated/candidate_pairs.parquet")
        return results_df

In [3]:
# Run the pair generator
sim = PairGenerator("data/processed/cleaned_yellow_tripdata_2025-08.parquet", "config/dev_config.yaml")
sim.filter_commuting_window()
sim.split_drivers_and_riders()
pairs_df = sim.generate_candidate_pairs(limit=500)
pairs_df.head()

Loading data from data/processed/cleaned_yellow_tripdata_2025-08.parquet...


Filtering for commute windows: 7:00-10:00 + 16:00-19:00
Downsampling to 25.0% for development.
Total trips in window: 268005
Assigned 133781 Drivers and 134224 Riders.
Starting candidate pair generation...


Processed 500 drivers. Feasible pairs: 1756
Generating negative labels from genuinely infeasible pairs...


Generated 1756 negative pairs from 2024 attempts.
Generated 3512 total labeled pairs (Balanced 1s and 0s).
Saved to data/simulated/candidate_pairs.parquet


,driver_id,rider_id,driver_orig_dist,detour_distance_km,overlap_ratio,time_diff_mins,driver_approx_fare,rider_approx_fare,fare_ratio,pickup_hour,pickup_dow,shareability_count,label
0,2693734,8893,2.258157,2.264851,0.499260,1.116667,11.9,9.3,0.781513,7,4,19,1
1,2693734,8700,2.258157,2.438869,0.480763,1.150000,11.9,7.2,0.605042,7,4,19,1
2,2693734,7357,2.258157,2.059258,0.523034,1.666667,11.9,5.1,0.428571,7,4,19,0
3,2693734,8816,2.258157,2.686267,0.456708,2.616667,11.9,7.2,0.605042,7,4,19,1
4,2693734,7245,2.258157,2.362064,0.488755,3.733333,11.9,7.9,0.663866,7,4,19,1
